# Make animated plots of data - OHC

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from celluloid import Camera
from IPython.display import HTML
from matplotlib.colors import Normalize

COUNTRIES_URL = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
WORLD = gpd.read_file(COUNTRIES_URL).rename({"ADMIN": "name"}, axis="columns")

## Load data

In [ ]:
df = pd.read_csv("Data/ME4OH_EN411_OFAM3/WORLD_1993_2014.csv")
# Convert dates to datetime, save year month and year indication columns (aids plotting)
df['Date'] = pd.to_datetime(df['Date'])
df['year_month'] = df['Date'].dt.to_period('M')
df['year'] = df['Date'].dt.to_period('Y')
df

In [ ]:
month_list = pd.date_range('1993-1-1','2014-12-31', freq='MS').strftime('%m/%Y').tolist()
year_list = pd.date_range('1993-1-1','2014-12-31', freq='YS').strftime('%Y').tolist()

### OHC world map

NOTE: Colourbar does not work in this plot, need to work this out later

Maybe try: https://stackoverflow.com/questions/18195758/set-matplotlib-colorbar-size-to-match-graph
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im, cax=cax)

In [ ]:
plt.rcParams.update({'font.size': 15})
fig, ax = plt.subplots(figsize=(15, 9))
basin = None
camera = Camera(fig)

fig.tight_layout(pad=2.5)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_facecolor("lightblue")

ax.set_xlim([-180, 180])
ax.set_ylim([-90, 90])

# Create a shared normalization object
norm = Normalize(vmin= min(df.OHC), vmax= max(df.OHC))
cmap = plt.cm.plasma   # Define the colormap
# s = ax.scatter(df.Longitude, df.Latitude, c=df.OHC, s=1, cmap='plasma', norm=norm, alpha=0)
# cbar = plt.colorbar(s, ax=ax, cmap=cmap, norm=norm)
# cbar.set_label('OHC $(Jm^{−2})$')
# cbar_axis = cbar.ax

for title,data in df.groupby('year'):
    # Plot OHC map
    # fig.clear()
    # ax = fig.add_subplot(111, aspect='equal', autoscale_on=False)#, xlim=(0, 1), ylim=(0, 1))
    # cbar_axis.cla()
    
    # ax.clear()
    WORLD.plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.75)  # Add countries
    s = ax.scatter(data.Longitude, data.Latitude, c=data.OHC, s=1, cmap='plasma', norm=norm)  # Plot OHC observations
    # cbar = fig.colorbar(s, cax=cax, cmap=cmap, norm=norm)
    # cbar.set_label('OHC $(Jm^{−2})$')
    ax.text(0.5, 1.01, title, transform=ax.transAxes)
    camera.snap() 
    
plt.close()
animation = camera.animate()  # animation ready
HTML(animation.to_html5_video())

In [ ]:
animation.save('images/world_ohc_year.mp4')

## OHC North Atlantic map

In [ ]:
regions = ['Southern Europe', 'Northern Africa', 'Western Africa', 'Western Europe', 'Northern Europe', 'Central America', 'South America', 'Northern America', 'Caribbean']
NA = WORLD[WORLD['SUBREGION'].isin(regions)]

In [ ]:
plt.rcParams.update({'font.size': 15})
na_df = pd.read_csv("Data/ME4OH_EN411_OFAM3/NA_1993_2014.csv")

# Convert dates to datetime, save year month and year indication columns (aids plotting)
na_df['Date'] = pd.to_datetime(na_df['Date'])
na_df['year_month'] = na_df['Date'].dt.to_period('M')
na_df['year'] = na_df['Date'].dt.to_period('Y')
na_df

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
camera = Camera(fig)

fig.tight_layout(pad=1.5)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_facecolor("lightblue")

ax.set_xlim([-80, 0])
ax.set_ylim([0, 60])

# Create a shared normalization object
norm = Normalize(vmin= min(na_df.OHC), vmax= max(na_df.OHC))
cmap = plt.cm.plasma   # Define the colormap

for title,data in na_df.groupby('year'):
    # Plot OHC map
    NA.plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.75)  # Add countries
    s = ax.scatter(data.Longitude, data.Latitude, c=data.OHC, s=1, cmap='plasma', norm=norm)  # Plot OHC observations
    ax.text(0.5, 1.01, title, transform=ax.transAxes)
    camera.snap()

plt.close()
animation = camera.animate()  # animation ready
HTML(animation.to_html5_video())

In [ ]:
animation.save('images/NA_ohc_year.mp4')

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
camera = Camera(fig)

fig.tight_layout(pad=1.5)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_facecolor("lightblue")

ax.set_xlim([-80, 0])
ax.set_ylim([0, 60])

# Create a shared normalization object
norm = Normalize(vmin= min(na_df.OHC), vmax= max(na_df.OHC))
cmap = plt.cm.plasma   # Define the colormap

for title,data in na_df.groupby('year_month'):
    # Plot OHC map
    NA.plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.75)  # Add countries
    s = ax.scatter(data.Longitude, data.Latitude, c=data.OHC, s=1, cmap='plasma', norm=norm)  # Plot OHC observations
    ax.text(0.5, 1.01, title, transform=ax.transAxes)

plt.close()
animation = camera.animate()  # animation ready
HTML(animation.to_html5_video())

In [ ]:
animation.save('images/NA_ohc_months.mp4')

## Platforms over time

In [ ]:
# Filter by platforms
platforms = {
    "Argo": df.loc[df['Project name'].str.contains('ARGO')].sort_values('Date').reset_index(drop=True),
    "WOD": df.loc[df['Project name'].str.contains('WOD')].sort_values('Date').reset_index(drop=True),
    "GTSPP": df.loc[df['Project name'].str.contains('GTSPP')].sort_values('Date').reset_index(drop=True),
    "ABSO":  df.loc[df['Project name'].str.contains('ASBO')].sort_values('Date').reset_index(drop=True),
}

colours = [ "#631ff3", "#ffb000", "#fe5100", "#1589e8", "#dc267f"]  # IBM colourblind palette

### Years

In [ ]:
plt.rcParams.update({'font.size': 15})
fig, ax = plt.subplots(figsize=(12, 6))
camera = Camera(fig)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_facecolor("lightblue")

ax.set_xlim([-180, 180])
ax.set_ylim([-90, 90])
legend=True

for year in year_list:
    # Plot OHC map
    WORLD.plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.75)  # Add countries
    for i, (name, data) in enumerate(platforms.items()):
        year_data = data.loc[data["year"] == year]
        ax.scatter(year_data.Longitude, year_data.Latitude, s=1, label=name, c=colours[i])
    ax.text(0.5, 1.01, year, transform=ax.transAxes)
    if legend:  # Only plot legend for first graph
        ax.legend(loc='upper right')
        legend=False
    camera.snap()

plt.close()
animation = camera.animate()  # animation ready
HTML(animation.to_html5_video())

In [ ]:
animation.save('images/world_platform_years.mp4')

### Months

In [ ]:
plt.rcParams.update({'font.size': 15})
fig, ax = plt.subplots(figsize=(12, 6))
camera = Camera(fig)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_facecolor("lightblue")

ax.set_xlim([-180, 180])
ax.set_ylim([-90, 90])
legend=True

for month in month_list:
    # Plot OHC map
    WORLD.plot(ax=ax, color='lightgray', edgecolor='gray', linewidth=0.75)  # Add countries
    for i, (name, data) in enumerate(platforms.items()):
        month_data = data.loc[data["year_month"] == month]
        ax.scatter(month_data.Longitude, month_data.Latitude, s=1, label=name, c=colours[i])
    ax.text(0.5, 1.01, month, transform=ax.transAxes)
    if legend:  # Only plot legend for first graph
        ax.legend(loc='upper right')
        legend=False
    camera.snap()

plt.close()
animation = camera.animate()  # animation ready
HTML(animation.to_html5_video())

In [ ]:
animation.save('images/world_platform_months.mp4')